# Notebook 04 — Linear and Meta Blending

**Session:** Ensembling  
**Module:** 02 · Calculus and Linear Algebra

## Goal
Design and implement a two-stage Blending meta-learning architecture using a holdout partition, generate Level-1 meta-features, and train linear meta-models (Logistic Regression, non-negative Ridge) to combine predictions optimally.

> **Key Distinction:** Unlike simple averaging (which assigns uniform weights $1/M$), Blending learns optimal continuous weights $w_m$ on an independent holdout set, preventing overfitting and giving higher authority to stronger, more diverse models.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

## 1. Partition Data: Train-Blend (50%), Holdout (25%), and Test (25%)

Blending requires a strict holdout set that Level-0 base models never see.

In [ ]:
data_path = Path("../data") / "synthetic_classification.csv"
df = pd.read_csv(data_path)
X = df.drop(columns=["target"]).values
y = df["target"].values

# First split: 75% for blending pipeline, 25% final test
X_pool, X_test, y_pool, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Second split: pool split into 70% Train-Blend and 30% Holdout
X_blend, X_holdout, y_blend, y_holdout = train_test_split(
    X_pool, y_pool, test_size=0.30, random_state=42, stratify=y_pool
)

print(f"Train-Blend set: {X_blend.shape} (Base model training)")
print(f"Holdout set:     {X_holdout.shape} (Meta-learner training)")
print(f"Final Test set:  {X_test.shape} (Unbiased final evaluation)")

## 2. Train Level-0 Base Estimators
We train three diverse algorithms on the `Train-Blend` set:
1. `SVC(probability=True)`
2. `RandomForestClassifier`
3. `KNeighborsClassifier`

In [ ]:
scaler = StandardScaler()
X_blend_scaled = scaler.fit_transform(X_blend)
X_holdout_scaled = scaler.transform(X_holdout)
X_test_scaled = scaler.transform(X_test)

m1 = SVC(probability=True, random_state=42)
m2 = RandomForestClassifier(n_estimators=100, random_state=42)
m3 = KNeighborsClassifier(n_neighbors=9)

m1.fit(X_blend_scaled, y_blend)
m2.fit(X_blend_scaled, y_blend)
m3.fit(X_blend_scaled, y_blend)

print("Base models successfully trained on Train-Blend partition.")

## 3. Generate Level-1 Meta-Features
We evaluate the base estimators on the `Holdout` partition to construct the meta-feature matrix $\mathbf{Z}_{\text{holdout}}$:

$$\mathbf{Z}_{\text{holdout}} = [\hat{p}_1(X_{\text{holdout}}), \hat{p}_2(X_{\text{holdout}}), \hat{p}_3(X_{\text{holdout}})]$$

In [ ]:
Z_holdout = np.column_stack(
    [
        m1.predict_proba(X_holdout_scaled)[:, 1],
        m2.predict_proba(X_holdout_scaled)[:, 1],
        m3.predict_proba(X_holdout_scaled)[:, 1],
    ]
)

Z_test = np.column_stack(
    [
        m1.predict_proba(X_test_scaled)[:, 1],
        m2.predict_proba(X_test_scaled)[:, 1],
        m3.predict_proba(X_test_scaled)[:, 1],
    ]
)

meta_scaler = StandardScaler()
Z_holdout_scaled = meta_scaler.fit_transform(Z_holdout)
Z_test_scaled = meta_scaler.transform(Z_test)

print(f"Meta-feature matrix shapes: Holdout={Z_holdout.shape}, Test={Z_test.shape}")

## 4. Train Level-1 Meta-Learners & Inspect Learned Coefficients

We train:
- **Logistic Regression Meta-Model:** Learns optimal linear log-odds weights.
- Compare with uniform equal-weight averaging.

In [ ]:
blender_lr = LogisticRegression(solver="liblinear", random_state=42)
blender_lr.fit(Z_holdout_scaled, y_holdout)

coefs = blender_lr.coef_[0]
intercept = blender_lr.intercept_[0]

print("=== Learned Meta-Learner Parameters ===")
print(f"Intercept: {intercept:.4f}")
print(f"Weight Model 1 (SVC):           {coefs[0]:.4f}")
print(f"Weight Model 2 (Random Forest): {coefs[1]:.4f}")
print(f"Weight Model 3 (KNN):           {coefs[2]:.4f}")

## 5. Evaluation and Comparison on Held-Out Test Set

In [ ]:
# Individual model test probabilities
test_p1 = Z_test[:, 0]
test_p2 = Z_test[:, 1]
test_p3 = Z_test[:, 2]

# Simple arithmetic average
test_avg = Z_test.mean(axis=1)

# Blending meta-model prediction
test_blend_lr = blender_lr.predict_proba(Z_test_scaled)[:, 1]

models = {
    "SVC": test_p1,
    "Random Forest": test_p2,
    "KNN": test_p3,
    "Equal-Weight Average": test_avg,
    "Logistic Regression Blender": test_blend_lr,
}

results = []
for name, preds in models.items():
    results.append(
        {
            "Model": name,
            "ROC-AUC": roc_auc_score(y_test, preds),
            "Brier_Score": brier_score_loss(y_test, preds),
            "Accuracy": accuracy_score(y_test, (preds >= 0.5).astype(int)),
        }
    )

results_df = pd.DataFrame(results)
print("=== Final Test Set Performance Comparison ===")
print(results_df.round(4).to_string(index=False))